# Analyse ad hoc — pilotage Vitalab

Notebook utilisé manuellement par l'analyste data pour extraire des résultats, produire des indicateurs et générer des exports pour l'équipe pilotage. Travaille directement sur la base et le bucket avec les identifiants partagés de l'équipe.

In [ ]:
import pandas as pd
import psycopg2

# Connexion directe à la base avec les identifiants de l'équipe
conn = psycopg2.connect(
    host="db",
    dbname="vitalab",
    user="svc_pipeline",
    password="V1talab-Prod-2024!",
)
df = pd.read_sql("SELECT * FROM staging.results_clean", conn)
df.head()

## Résultats anormaux par patient

In [ ]:
abnormal = df[df["result_flag"].isin(["anormal", "critique"])]
abnormal[[
    "patient_id",
    "patient_last_name",
    "patient_email",
    "analysis_label",
    "result_value",
    "result_flag",
]]

## Export pour l'équipe pilotage

Export nominatif déposé pour le pilotage (identité + résultats).

In [ ]:
abnormal.to_csv("exports/anomalies_pilotage.csv", index=False)

## Dépôt de l'export sur le bucket objet

Les analystes se connectent au bucket avec les clés d'accès partagées de l'équipe (les mêmes que le pipeline) pour récupérer les exports existants et déposer les nouveaux.

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="AKIA_VITALAB_EXAMPLE",
    aws_secret_access_key="wJalrXUtnFEMI_vitalab_example_key",
)
# Récupérer l'export du pipeline puis déposer l'export d'anomalies
s3.download_file("vitalab-data", "exports/results_dump.csv", "results_dump.csv")
s3.upload_file("exports/anomalies_pilotage.csv", "vitalab-data", "exports/anomalies_pilotage.csv")